In [1]:
# 01_data_preprocessing.ipynb
# KNHANES 2020-2024 merge, target encoding, 4-class risk labelling,
# English column renaming, and construction of the all-numeric modelling
# dataset used by every downstream notebook.
#
# Inputs  (place under ../data/):
#     hn20_all.sas7bdat ... hn24_all.sas7bdat   (KDCA raw survey files)
# Outputs (written to ../data/):
#     knhanes_comorbid_2020_2024.csv   human-readable merged table
#     df_final.pkl                     all-numeric modelling frame
#     agent_config.pkl                 feature lists + metadata
#
# NOTE ON STUDY DESIGN: KNHANES is a repeated CROSS-SECTIONAL survey. Each
# annual cycle is an independent probability sample; there is no personal
# key linking a respondent across years. No longitudinal follow-up is
# therefore possible from these data, which bounds every transition-based
# claim in this project to a within-model, cross-sectional interpretation.

import os
import json
import joblib
import numpy as np
import pandas as pd
import pyreadstat

DATA_DIR = os.path.join("..", "data")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 100)
pd.set_option("display.max_colwidth", None)

# ---------------------------------------------------------------------------
# 1. Load five KNHANES cycles (2020-2024)
# ---------------------------------------------------------------------------
frames = []
for yy in ["20", "21", "22", "23", "24"]:
    path = os.path.join(DATA_DIR, f"hn{yy}_all.sas7bdat")
    df_yy, _ = pyreadstat.read_sas7bdat(path)
    frames.append(df_yy)
    print(f"  hn{yy}_all: {df_yy.shape}")

# ---------------------------------------------------------------------------
# 2. Variable selection
# ---------------------------------------------------------------------------
KEY_COLS = ["ID", "year", "sex"]
CAT_COLS = [
    "HE_obe", "BO1_1", "BO1_2", "BO1_3", "BD1_11", "BD2_1", "BS3_1",
    "BE3_71", "BE3_75", "BE3_81", "BE3_91", "pa_aerobic", "L_BR_FQ",
    "BP1", "mh_stress", "incm", "ho_incm", "edu", "BH1",
]
NUM_COLS = [
    "HE_BMI", "HE_wc", "HE_wt",
    "N_EN", "N_CHO", "N_SUGAR", "N_NA", "N_FAT",
    "N_SFA", "N_TDF", "N_K", "N_PROT",
]
TARGET_COLS = ["HE_DM_HbA1c", "HE_HP"]
ALL_VARS = KEY_COLS + CAT_COLS + NUM_COLS + TARGET_COLS

df_total = pd.concat(
    [d[ALL_VARS].copy() for d in frames], axis=0
).reset_index(drop=True)
print("merged raw:", df_total.shape)

# ---------------------------------------------------------------------------
# 3. Target encoding
#    HE_DM_HbA1c: 1 (Normal) -> 0, 3 (Diabetes) -> 1
#    HE_HP      : 1 (Normal) -> 0, 4 (Hypertension) -> 1
# ---------------------------------------------------------------------------
df_total["HE_DM_HbA1c"] = df_total["HE_DM_HbA1c"].map({1: 0, 3: 1}).fillna(-999)
df_total["HE_HP"]       = df_total["HE_HP"].map({1: 0, 4: 1}).fillna(-999)

# ---------------------------------------------------------------------------
# 4. Integrated 4-class risk label
#    Class 0 Normal | Class 1 HTN-only | Class 2 DM-only | Class 3 Comorbid
# ---------------------------------------------------------------------------
conditions = [
    (df_total["HE_DM_HbA1c"] == 0) & (df_total["HE_HP"] == 0),
    (df_total["HE_DM_HbA1c"] == 0) & (df_total["HE_HP"] == 1),
    (df_total["HE_DM_HbA1c"] == 1) & (df_total["HE_HP"] == 0),
    (df_total["HE_DM_HbA1c"] == 1) & (df_total["HE_HP"] == 1),
]
df_total["integrated_target"] = np.select(conditions, [0, 1, 2, 3], default=np.nan)
df_total = df_total.dropna().reset_index(drop=True)
df_total["integrated_target"] = df_total["integrated_target"].astype(int)

print("\nRisk class distribution:")
print(df_total["integrated_target"].value_counts().sort_index())

# ---------------------------------------------------------------------------
# 5. Rename SAS variable codes -> English labels
# ---------------------------------------------------------------------------
ENGLISH_LABEL_DICT = {
    "ID": "ID", "year": "SurveyYear", "sex": "Sex",
    "HE_DM_HbA1c": "Diabetes", "HE_HP": "Hypertension",
    "integrated_target": "RiskClass",
    "HE_obe": "ObesityStatus", "BO1_1": "WeightChangeStatus",
    "BO1_2": "WeightLossAmount", "BO1_3": "WeightGainAmount",
    "BD1_11": "DrinkingFrequency", "BD2_1": "DrinkingAmount",
    "BS3_1": "SmokingStatus", "BE3_71": "VigorousActivity_Work",
    "BE3_75": "VigorousActivity_Leisure", "BE3_81": "ModerateActivity_Work",
    "BE3_91": "WalkingActivity", "pa_aerobic": "AerobicActivityRate",
    "L_BR_FQ": "BreakfastFrequency", "BP1": "StressLevel",
    "mh_stress": "StressAwarenessRate", "incm": "PersonalIncomeQuartile",
    "ho_incm": "HouseholdIncomeQuartile", "edu": "EducationLevel",
    "BH1": "HealthScreeningStatus",
    "HE_BMI": "BMI", "HE_wc": "WaistCirc", "HE_wt": "Weight",
    "N_EN": "Energy_kcal", "N_CHO": "Carb_g", "N_SUGAR": "Sugar_g",
    "N_NA": "Sodium_mg", "N_FAT": "Fat_g", "N_SFA": "SaturatedFat_g",
    "N_TDF": "Fiber_g", "N_K": "Potassium_mg", "N_PROT": "Protein_g",
}
df_total.rename(columns=ENGLISH_LABEL_DICT, inplace=True)

csv_path = os.path.join(DATA_DIR, "knhanes_comorbid_2020_2024.csv")
df_total.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"\n>>> merged CSV saved -> {csv_path}")

# ---------------------------------------------------------------------------
# 6. Feature definition and all-numeric modelling frame
# ---------------------------------------------------------------------------
NUM_FEATURES = [
    "BMI", "WaistCirc", "Weight",
    "Energy_kcal", "Carb_g", "Sugar_g", "Sodium_mg",
    "Fat_g", "SaturatedFat_g", "Fiber_g", "Potassium_mg", "Protein_g",
]
CAT_FEATURES = [
    "ObesityStatus", "WeightChangeStatus", "WeightLossAmount", "WeightGainAmount",
    "DrinkingFrequency", "DrinkingAmount", "SmokingStatus",
    "VigorousActivity_Work", "VigorousActivity_Leisure",
    "ModerateActivity_Work", "WalkingActivity", "AerobicActivityRate",
    "BreakfastFrequency", "StressLevel", "StressAwarenessRate",
    "PersonalIncomeQuartile", "HouseholdIncomeQuartile",
    "EducationLevel", "HealthScreeningStatus",
]
X_FEATURES = NUM_FEATURES + CAT_FEATURES
TARGET_COL = "RiskClass"

# Non-modifiable / administrative variables held fixed during DiCE search.
FIXED_FEATURES = [
    "StressLevel", "StressAwarenessRate",
    "PersonalIncomeQuartile", "HouseholdIncomeQuartile",
    "EducationLevel", "HealthScreeningStatus",
]
VARY_FEATURES = [f for f in X_FEATURES if f not in FIXED_FEATURES]

required_cols = X_FEATURES + [TARGET_COL, "Sex"]
df_final = df_total[required_cols].copy()
for col in df_final.columns:
    df_final[col] = pd.to_numeric(df_final[col], errors="coerce").fillna(0)
df_final = df_final.astype(float)

print(f"\ndf_final shape : {df_final.shape}")
print(f"Any string column: {(df_final.dtypes == 'object').any()}")

# ---------------------------------------------------------------------------
# 7. Persist artefacts for downstream notebooks
# ---------------------------------------------------------------------------
agent_config = {
    "X_features":     X_FEATURES,
    "num_features":   NUM_FEATURES,
    "cat_features":   CAT_FEATURES,
    "fixed_features": FIXED_FEATURES,
    "vary_features":  VARY_FEATURES,
    "target_col":     TARGET_COL,
}
joblib.dump(df_final,     os.path.join(DATA_DIR, "df_final.pkl"))
joblib.dump(agent_config, os.path.join(DATA_DIR, "agent_config.pkl"))
print(">>> df_final.pkl and agent_config.pkl saved to ../data/")


  hn20_all: (7359, 859)
  hn21_all: (7090, 864)
  hn22_all: (6265, 623)
  hn23_all: (6929, 629)
  hn24_all: (6997, 798)
merged raw: (34640, 36)

Risk class distribution:
integrated_target
0    6381
1    1370
2     649
3    1338
Name: count, dtype: int64

>>> merged CSV saved -> ..\data\knhanes_comorbid_2020_2024.csv

df_final shape : (9738, 33)
Any string column: False
>>> df_final.pkl and agent_config.pkl saved to ../data/
